#### T01 Production Quick Start

Enterprise-style onboarding: one YAML contract, logger at startup, inject into a small service, then log with `layer` + `context` (not raw f-strings).

`examples/config/tutorial_t01_enterprise_layers.yaml` — **Expect:** console (`app`, `error`) + `t01_app.jsonl`, `t01_audit.log`, `t01_error.jsonl`. <small>More: `examples/tutorials/notebooks/README.md`.</small>


**§0 — pip**  
<small>Optional if the package is already importable.</small>


In [1]:
%pip install -q "hydra-logger"
# Restart kernel if pip upgraded deps.


Note: you may need to restart the kernel to use updated packages.


**§1 — Setup**  
<small>Same as other tutorials: collapsed cell runs ``prime_notebook_workspace()`` via ``examples/tutorials/jupyter_workspace.py``. ``CONFIG_PATH`` is under ``examples/config/``. Details: `notebooks/README.md`.</small>


In [2]:
import importlib.util
import os
import tempfile
from pathlib import Path

def _resolved_notebook_cwd() -> Path:
    try:
        return Path.cwd().resolve()
    except FileNotFoundError:
        fb = Path(tempfile.gettempdir()).resolve()
        os.chdir(fb)
        return fb

def _load_jupyter_workspace_module():
    _starts: list[Path] = []
    _env = os.environ.get("HYDRA_LOGGER_REPO", "").strip()
    if _env:
        _starts.append(Path(_env).expanduser().resolve())
    _starts.append(_resolved_notebook_cwd())
    _seen: set[str] = set()
    for _s in _starts:
        for _b in (_s, *_s.parents):
            _jw = _b / "examples" / "tutorials" / "jupyter_workspace.py"
            try:
                _key = str(_jw.resolve())
            except (OSError, RuntimeError):
                continue
            if _key in _seen:
                continue
            _seen.add(_key)
            if _jw.is_file():
                _spec = importlib.util.spec_from_file_location("_hydra_tutorial_jw", _jw)
                assert _spec and _spec.loader
                _mod = importlib.util.module_from_spec(_spec)
                _spec.loader.exec_module(_mod)
                return _mod
    raise FileNotFoundError(
        "Could not find examples/tutorials/jupyter_workspace.py. Clone hydra-logger, set "
        "HYDRA_LOGGER_REPO to the clone (or any directory inside it), or start Jupyter with cwd "
        "inside that clone."
    )

repo_root = _load_jupyter_workspace_module().prime_notebook_workspace()


Repo root (cwd): /home/razvansavin/Projects/hydra-logger


**§2 — Imports + paths**  
<small>`CONFIG_PATH`, `LOG_DIR`.</small>


In [3]:
from dataclasses import dataclass
from typing import Any

from hydra_logger import create_sync_logger
from hydra_logger.config.loader import load_logging_config

CONFIG_PATH = repo_root / "examples" / "config" / "tutorial_t01_enterprise_layers.yaml"
LOG_DIR = repo_root / "examples" / "logs" / "notebooks"


**§3 — Peek config (optional)**  
<small>Validate YAML before constructing the logger.</small>


In [4]:
cfg = load_logging_config(CONFIG_PATH, strict_unknown_fields=True)
print("layers:", list(cfg.layers.keys()))
print("default_level:", cfg.default_level)


layers: ['app', 'audit', 'error']
default_level: INFO


**§4 — Application (DI)**  
<small>Inject `logger`; use `layer` / `context` / `extra`.</small>


In [5]:
@dataclass
class PaymentRequest:
    order_id: str
    user_id: str
    amount: float
    currency: str = "USD"


class PaymentService:
    def __init__(self, logger: Any) -> None:
        self.logger = logger

    def process(self, req: PaymentRequest) -> dict[str, Any]:
        self.logger.info(
            "payment request received",
            layer="app",
            context={"order_id": req.order_id, "user_id": req.user_id},
            extra={"amount": req.amount, "currency": req.currency},
        )
        if req.amount <= 0:
            self.logger.warning(
                "invalid payment amount",
                layer="audit",
                context={"order_id": req.order_id, "user_id": req.user_id},
                extra={"amount": req.amount},
            )
            raise ValueError("Amount must be > 0")
        if req.amount > 10000:
            self.logger.error(
                "payment exceeds policy threshold",
                layer="error",
                context={"order_id": req.order_id, "user_id": req.user_id},
                extra={"amount": req.amount, "threshold": 10000},
            )
            raise RuntimeError("Risk policy blocked this payment")
        self.logger.info(
            "payment approved",
            layer="app",
            context={"order_id": req.order_id},
            extra={"status": "approved"},
        )
        return {"ok": True, "order_id": req.order_id, "status": "approved"}


**§5 — Logger + scenarios**  
<small>`create_sync_logger` from YAML, then exercise paths.</small>


In [6]:
with create_sync_logger(
    config_path=CONFIG_PATH,
    strict_unknown_fields=True,
    name="t01-tutorial",
) as logger:
    svc = PaymentService(logger)
    svc.process(PaymentRequest(order_id="ORD-1001", user_id="U-1", amount=99.5))
    svc.process(PaymentRequest(order_id="ORD-1004", user_id="U-4", amount=42.0))
    svc.process(PaymentRequest(order_id="ORD-1005", user_id="U-5", amount=250.0))
    logger.info(
        "batch settlement completed",
        layer="app",
        context={"batch_id": "B-09"},
        extra={"items": 3},
    )
    try:
        svc.process(PaymentRequest(order_id="ORD-1002", user_id="U-2", amount=0))
    except ValueError:
        pass
    try:
        svc.process(PaymentRequest(order_id="ORD-1003", user_id="U-3", amount=15000))
    except RuntimeError:
        pass


| 2026-03-21 17:17:51 | INFO | app | payment request received
| 2026-03-21 17:17:51 | INFO | app | payment approved
| 2026-03-21 17:17:51 | INFO | app | payment request received
| 2026-03-21 17:17:51 | INFO | app | payment approved
| 2026-03-21 17:17:51 | INFO | app | payment request received
| 2026-03-21 17:17:51 | INFO | app | payment approved
| 2026-03-21 17:17:51 | INFO | app | batch settlement completed
| 2026-03-21 17:17:51 | INFO | app | payment request received
| 2026-03-21 17:17:51 | INFO | app | payment request received
| 2026-03-21 17:17:51 | ERROR | error | payment exceeds policy threshold


**§6 — File tails**  
<small>Re-run after §5.</small>


In [7]:
print("File output (last lines):")
for _name in ("t01_app.jsonl", "t01_audit.log", "t01_error.jsonl"):
    _p = LOG_DIR / _name
    print(f"--- {_name} ---")
    if _p.is_file():
        _lines = _p.read_text(encoding="utf-8").splitlines()
        for _line in _lines[-8:]:
            print(_line)
    else:
        print("(missing — run §5 first)")


File output (last lines):
--- t01_app.jsonl ---
{"timestamp":"2026-03-21 17:17:51","level":20,"level_name":"INFO","message":"payment approved","logger_name":"t01-tutorial","layer":"app","file_name":"1940976155.py","function_name":"PaymentService.process","line_number":36,"extra":{"status":"approved"}}
{"timestamp":"2026-03-21 17:17:51","level":20,"level_name":"INFO","message":"payment request received","logger_name":"t01-tutorial","layer":"app","file_name":"1940976155.py","function_name":"PaymentService.process","line_number":14,"extra":{"amount":42.0,"currency":"USD"}}
{"timestamp":"2026-03-21 17:17:51","level":20,"level_name":"INFO","message":"payment approved","logger_name":"t01-tutorial","layer":"app","file_name":"1940976155.py","function_name":"PaymentService.process","line_number":36,"extra":{"status":"approved"}}
{"timestamp":"2026-03-21 17:17:51","level":20,"level_name":"INFO","message":"payment request received","logger_name":"t01-tutorial","layer":"app","file_name":"194097615

**Iterate:** edit YAML at `CONFIG_PATH`, re-run **§3** → **§5** → **§6**. <small>`notebooks/README.md`</small>
